# L12d Lab: Overestimation and Double DQN
In this lab, we look at a flaw of Q-learning and DQN: their values tend to come out too high. We see where this __overestimation__ comes from and watch it steer learners to the worse action. Then we test two fixes, Double Q-learning and Double DQN.

__Background:__ The Q-learning target uses the largest estimated value of the next state. Each estimate is noisy, because it is learned from a few random rewards. The largest of several noisy estimates tends to be one whose noise happened to be positive.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Explain where overestimation comes from:__ The largest of several noisy estimates is too large on average, even when each estimate is right on average. We'll measure this bias and remove it by choosing with one set of samples and scoring with another.
> * __Compare Q-learning with Double Q-learning:__ Overestimation can make Q-learning prefer an action whose true value is worse. We'll watch this happen on a problem with two states and see Double Q-learning avoid it.
> * __Test Double DQN:__ Double DQN uses the target network as its second estimate, which costs nothing extra but is not independent of the Q-network. We'll measure how much of the bias it removes and how that depends on how often we copy the target network.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

The estimates, the two-state problem, and the four learners are in [src/TwoState.jl](src/TwoState.jl), and the figure is in [src/Visualize.jl](src/Visualize.jl). This code is complete. There is nothing to edit in this lab.
___

## Task 1: The Largest of Several Noisy Estimates
In this task, we measure how far the largest of several noisy estimates lands above the true largest value.

Suppose an agent has 10 actions, and every action has the same true value, $-0.1$. The agent does not know this. It estimates each value as the average of 10 random rewards, each drawn from a normal distribution with mean $-0.1$ and standard deviation 1.

The Q-learning target uses the largest of these estimates. The true largest value is $-0.1$.

[The `maxbias(...)` function](src/TwoState.jl) repeats this experiment 10,000 times and returns the average of two estimates of the largest value:
* The __single estimate__ takes the largest of the 10 averages.
* The __double estimate__ splits the 10 rewards of each action into two halves of 5. The first halves choose the action with the largest average. The second halves report that action's average.

__Prediction:__ Averaged over the 10,000 trials, where will the single estimate land: {below $-0.1$; about $-0.1$; or above $-0.1$}?

In [ ]:
estimates = maxbias(nactions = 10, nsamples = 10, μ = -0.1, trials = 10_000, seed = 1234);
println("single estimate: ", round(estimates.single, digits = 3))
println("double estimate: ", round(estimates.double, digits = 3))
println("true largest value: ", -0.1)

__What do we see?__ The single estimate averages about 0.39, nearly half a unit above the true value of $-0.1$. The double estimate averages $-0.10$, the true value.

The largest average is usually an action whose rewards happened to be high. The double estimate scores that action with its other five rewards, whose noise is independent of the first five. Its luck does not carry over.

__Prediction:__ Suppose the agent has 100 actions instead of 10. Will the single estimate move {closer to $-0.1$; stay about the same; or move further above $-0.1$}?

In [ ]:
let
    # your turn: change the setting and rerun (try other values) -
    nactions_new = 100; # the number of actions
    nsamples_new = 10; # the number of rewards per action (an even number)

    # repeat the experiment and report both estimates -
    estimates_new = maxbias(nactions = nactions_new, nsamples = nsamples_new, μ = -0.1,
        trials = 10_000, seed = 1234);
    println(nactions_new, " actions, ", nsamples_new, " rewards each: single ",
        round(estimates_new.single, digits = 3), ", double ", round(estimates_new.double, digits = 3))
end

__What do we see?__ With 100 actions, the single estimate averages about 0.69. More actions give the noise more chances to produce one lucky estimate. The double estimate stays at $-0.10$.

### Things to think about

__Question:__ Rerun the cell with `nactions_new = 1`, then with `nactions_new = 10` and `nsamples_new = 100`. How far above $-0.1$ is the single estimate in each run? Which two settings keep overestimation small? Briefly explain.

___

## Task 2: Q-Learning versus Double Q-Learning
In this task, we watch overestimation steer Q-learning to the worse action on a small problem from [Sutton and Barto (2018), Example 6.7](http://incompleteideas.net/book/the-book-2nd.html).

The problem has two states, A and B:
* Every episode starts in A, which has two actions. The action "right" ends the episode with reward 0. The action "left" moves to B with reward 0.
* B has 10 actions. Each one ends the episode with a reward drawn from a normal distribution with mean $-0.1$ and standard deviation 1.

Going left earns $-0.1$ on average, and going right earns 0. The better action in A is "right".

The agents explore with $\epsilon=0.1$. An exploring step in A picks "left" half the time. Even an agent that rates "right" higher then chooses "left" in about 5 percent of the episodes.

[The `twostate_qlearning(...)` function](src/TwoState.jl) trains 1,000 agents for 300 episodes each, with learning rate $\alpha=0.1$ and no discounting. It returns the fraction of the agents that chose "left" in each episode. With `double = true`, the agents use Double Q-learning:

> __Double Q-learning__ ([van Hasselt, 2010](https://papers.nips.cc/paper_files/paper/2010/hash/091d584fced301b442654dd8c23b3fc9-Abstract.html)) keeps two tables, $Q_{1}$ and $Q_{2}$, and acts on their sum. After each step, a coin flip picks the table to update. When it picks $Q_{1}$, the target of "left" uses the action that $Q_{1}$ rates best in B, scored by $Q_{2}$:
>
> $$y = r + Q_{2}\left(B,\;\arg\max_{b}Q_{1}(B,b)\right)$$
>
> When it picks $Q_{2}$, the two tables swap roles.

This is the double estimate of Task 1, with the two tables in place of the two halves of the rewards.

__Prediction:__ Over the first 20 episodes, will the share of Q-learning agents that choose "left" {fall toward 5 percent; stay near 50 percent; or rise above 50 percent}?

In [ ]:
left_q = twostate_qlearning(double = false, episodes = 300, runs = 1000, nB = 10, μ = -0.1,
    α = 0.1, ϵ = 0.1, seed = 1234); # Q-learning
left_double = twostate_qlearning(double = true, episodes = 300, runs = 1000, nB = 10, μ = -0.1,
    α = 0.1, ϵ = 0.1, seed = 1234); # Double Q-learning
plot_left(left_q, left_double; label_one = "Q-learning", label_two = "Double Q-learning", ϵ = 0.1)

__What do we see?__ The share of Q-learning agents that choose "left" rises to 94 percent in episode 20. Almost every agent prefers the worse action. The share then falls slowly, to about 12 percent by episode 300.

The Double Q-learning agents never go above about 51 percent, the coin flip of the first episodes. Their share falls below 10 percent by episode 82.

Early on, each value in B rests on a few rewards. The largest of the 10 is then often above 0, and "left" looks better than "right". Double Q-learning scores the chosen action with the other table, whose noise is independent.

__Prediction:__ Suppose B has 100 actions instead of 10. How will the Q-learning agents behave over the 300 episodes: {as before; worse, with "left" chosen longer; or better}?

In [ ]:
let
    # your turn: change the problem and rerun (try other values) -
    nB_new = 100; # the number of actions in B

    # train both kinds of agents and plot their choices -
    left_q_new = twostate_qlearning(double = false, nB = nB_new, seed = 1234);
    left_double_new = twostate_qlearning(double = true, nB = nB_new, seed = 1234);
    plot_left(left_q_new, left_double_new; label_one = "Q-learning", label_two = "Double Q-learning",
        ϵ = 0.1, title = "$(nB_new) actions in B") # return
end

__What do we see?__ With 100 actions in B, about 95 percent of the Q-learning agents still choose "left" in episode 300. They never recover in this run. The Double Q-learning agents behave much as before and fall to about 7 percent.

### Things to think about

__Question:__ Rerun the cell with `nB_new = 1`. Does the share of Q-learning agents that choose "left" still rise above 50 percent? Compare the two curves. What does this run tell us about where the bias comes from? Briefly explain.

___

## Task 3: DQN versus Double DQN
In this task, we train DQN agents on the same problem and test the Double DQN target from the lecture.

Our DQN agents use the algorithm of the lecture with a linear Q-network whose features are one-hot. There is one weight per state and action, the same numbers as a table. Each agent works as follows:
* It stores every transition in a replay buffer.
* After each step, it takes one gradient step with learning rate $\alpha=0.5$ on a random mini-batch of 8 stored transitions.
* Every $C=25$ steps, it copies the Q-network to the target network.

Double DQN changes only the target of "left". The Q-network picks the best action in B, and the target network scores it:

$$
\begin{equation*}
y = r + Q\left(B,\;\arg\max_{b}Q(B,b;\theta);\;\theta^{-}\right)
\end{equation*}
$$

where $\theta$ are the weights of the Q-network and $\theta^{-}$ are the weights of the target network. The target network plays the role of the second table. DQN already has it, at no extra cost. But it is a copy of the Q-network from at most $C$ steps ago, trained on mostly the same rewards.

[The `twostate_dqn(...)` function](src/TwoState.jl) trains 1,000 DQN agents, or Double DQN agents with `double = true`, and returns the fraction that chose "left" in each episode.

__Prediction:__ Will Double DQN remove the bias as well as Double Q-learning did: {yes; partly; or not at all}?

In [ ]:
left_dqn = twostate_dqn(double = false, episodes = 300, runs = 1000, nB = 10, μ = -0.1,
    α = 0.5, ϵ = 0.1, batchsize = 8, warmup = 8, C = 25, seed = 1234); # DQN
left_ddqn = twostate_dqn(double = true, episodes = 300, runs = 1000, nB = 10, μ = -0.1,
    α = 0.5, ϵ = 0.1, batchsize = 8, warmup = 8, C = 25, seed = 1234); # Double DQN
plot_left(left_dqn, left_ddqn; label_one = "DQN", label_two = "Double DQN", ϵ = 0.1)

__What do we see?__ Double DQN helps, but much less than Double Q-learning did. Its share of "left" peaks at 86 percent, against 94 percent for DQN. In episode 300, about 54 percent of the Double DQN agents still choose "left", against about 69 percent of the DQN agents.

The target network learned from mostly the same rewards as the Q-network. Its noise is not independent, and the action that looks lucky to one often looks lucky to the other.

__Prediction:__ Suppose we copy the Q-network to the target network after every step, `C_new = 1`. What happens to the gap between the two curves: {it grows; it stays the same; or it disappears}?

In [ ]:
let
    # your turn: change the copy interval and rerun (try other values) -
    C_new = 1; # the number of steps between copies of the Q-network to the target network

    # train both kinds of agents and plot their choices -
    left_dqn_new = twostate_dqn(double = false, C = C_new, seed = 1234);
    left_ddqn_new = twostate_dqn(double = true, C = C_new, seed = 1234);
    plot_left(left_dqn_new, left_ddqn_new; label_one = "DQN", label_two = "Double DQN",
        ϵ = 0.1, title = "copy every $(C_new) steps") # return
end

__What do we see?__ With `C_new = 1`, the two curves are the same. The target network is then identical to the Q-network, and the Double DQN target equals the DQN target.

### Things to think about

__Question:__ Rerun the cell with `C_new = 100`. How does the gap between DQN and Double DQN change? What happens in the first 60 or so episodes, before the first copy? What do we pay for copying the target network less often? Briefly explain.

___

## Summary
In this lab, we measured overestimation, watched it steer Q-learning and DQN to the worse action, and tested Double Q-learning and Double DQN.

> __Key Takeaways:__
>
> * __The largest of several noisy estimates is too large:__ With 10 actions whose true values were all −0.1, the largest of their estimates averaged about 0.39. Choosing with one half of the rewards and scoring with the other half removed this bias.
>
> * __Overestimation can make Q-learning prefer the worse action:__ Early in training, almost every Q-learning agent preferred "left", whose noisy estimates looked best, although "right" was better. Double Q-learning, with two tables that score each other's choices, avoided this.
>
> * __Double DQN removes only part of the bias:__ The target network is a recent copy of the Q-network, not an independent estimate. Double DQN reduced the bias, and copying the target network less often reduced it more, at the price of slower learning.

Overestimation comes from taking a maximum over noisy estimates. Any learner that bootstraps from the largest estimated value, tabular or deep, inherits it.

___